## Structure of this notebook

- **(1) Preparations**
  - **(1.1)** Import of all necessary packages
  - **(1.2)** Loading raw excel data of population and spatial data of neighbourhoods

- **(2) Process raw excel data to DataFrame**
  - **(2.1)** Read separate yearly excel sheets, clean unnecessary rows and columns
  - **(2.2)** Put all years into one DataFrame
  - **(2.3)** Save as cleaned .csv to data-folder

- **(3) Build GeoDataFrame of neighbourhood-population with geometries**

- **(4) Save as cleaned .gpkg to data-folder**

In [1]:
# (1) Preparations

# (1.1) Import of all necessary packages
import pandas as pd
import geopandas as gpd
from shapely import wkt

# (1.2) Loading raw excel data of population and spatial data of neighbourhoods
pop_excel = pd.ExcelFile(
    "../data/raw/BEV321T3211_auslaendische-Wohnbevoelkerung_Bevoelkerung_nach-Herkunft-Stadtkreis-Stadtquartier.xlsx"
)
quartiere_gdf = gpd.read_file("../data/raw/quartiere.gpkg", layer="stzh.adm_statistische_quartiere_v")

print("Data loaded successfully\n")

Data loaded successfully



In [2]:
# (2) Process raw excel data to DataFrame
years = range(2013, 2026)

# (2.1) Read separate yearly excel sheets, clean unnecessary rows and columns
all_years = []

for year in years:

    # Read one sheet (= one year)
    year_df = pd.read_excel(pop_excel, sheet_name=str(year), skiprows=8)

    # Clean yearly population data
    year_df = (
        year_df
        .rename(columns={"Unnamed: 0": "quartier"})
        [["quartier", "Total"]]
    )

    # Remove city total and district rows (Keep, where not "Ganze Stadt" and "Kreis...")
    year_df = year_df[
        (year_df["quartier"] != "Ganze Stadt")
        & (~year_df["quartier"].str.startswith("Kreis"))
    ]

    # Rename population column to corresponding year
    year_df = year_df.rename(columns={"Total": str(year)})

    # Store cleaned dataframe
    all_years.append(year_df)

# (2.2) Putting all years into one DataFrame
pop_df = all_years[0]

for df in all_years[1:]:
    pop_df = pop_df.merge(df, on="quartier")

# (2.3) Save as CSV
output_file = "../data/cleaned/quartier_population_2013_2025.csv"
pop_df.to_csv(output_file, index=False)

print("CSV successfully created!")

CSV successfully created!


In [3]:
# (3) Build GeoDataFrame of neighbourhood-population with geometries
# Join population data to spatial data of neighbourhoods
pop_quartiere_gdf = quartiere_gdf.merge(
    pop_df,
    left_on="qname",
    right_on="quartier",
    how="left"
)

# define the parameters for the resulting GeoDataFrame
pop_quartiere_gdf = gpd.GeoDataFrame(
    pop_quartiere_gdf, 
    geometry="geometry",
    crs=quartiere_gdf.crs
)

# Delete the doubled column of neighbourhood names and columns with unnecessary information
pop_quartiere_gdf = pop_quartiere_gdf.drop(
    columns=["quartier", "objid", "objectid", "qnr", "kname", "knr"])

In [5]:
# (4) Save as cleaned .gpkg to data-folder
pop_quartiere_gdf.to_file(
    "../data/cleaned/pop_quartiere_2013_2025.gpkg",
    driver="GPKG"
)

print("GeoPackage successfully created!")

GeoPackage successfully created!
